In [1]:
%load_ext dotenv
%dotenv 

# What are we doing?

## Objectives 


* Build a data pipeline that downloads price data from the internet, stores it locally, transforms it into return data, and stores the feature set.
    - Getting the data.
    - Schemas and index in dask.

* Explore the parquet format.
    - Reading and writing parquet files.
    - Read datasets that are stored in distributed files.
    - Discuss dask vs pandas as a small example of big vs small data.
    
* Discuss the use of environment variables for settings.
* Discuss how to use Jupyter notebooks and source code concurrently. 
* Logging and using a standard logger.

## About the Data

+ We will download the prices for a list of stocks.
+ The source is Yahoo Finance and we will use the API provided by the library yfinance.


## Medallion Architecture

+ The architecture that we are thinking about is called Medallion by [DataBricks](https://www.databricks.com/glossary/medallion-architecture). It is an ELT type of thinking, although our data is well-structured.

![Medallion Architecture (DataBicks)](./images/02_medallion_architecture.png)

+ In our case, we would like to optimize the number of times that we download data from the internet. 
+ Ultimately, we will build a pipeline manager class that will help us control the process of obtaining and transforming our data.

![](./images/02_target_pipeline_manager.png)

# Download Data

Download the [Stock Market Dataset from Kaggle](https://www.kaggle.com/datasets/jacksoncrow/stock-market-dataset). Note that you may be required to register for a free account.

Extract all files into the directory: `./05_src/data/prices_csv/`

Your folder structure should include the following paths:

+ `05_src/data/prices_csv/etfs`
+ `05_src/data/prices_csv/stocks`


In [2]:
import pandas as pd
import os
import sys
from glob import glob

sys.path.append(os.getenv('SRC_DIR')) # Ensure the SRC_DIR is in the system path for module imports

from utils.logger import get_logger # Importing the logger utility from utils.logger

# _variable means its private to this module, a common practice in development
_logs = get_logger(__name__) # Initialize the logger with the current module's name

A few things to notice in the code chunk above:

+ Libraries are ordered from high-level to low-level libraries from the package manager (pip in this case, but could be conda, poetry, etc.)
+ The command `sys.path.append("../05_src/)` will add the `../05_src/` directory to the path in the Notebook's kernel. This way, we can use our modules as part of the notebook.
+ Local modules are imported at the end. 
+ The function `get_logger()` is called with `__name__` as recommended by the documentation.

Now, to load the historical price data for stocks and ETFs, we could use:

In [ ]:
import random

# Get a list of all stock price CSV files in the specified directory
stock_files = glob(os.path.join(os.getenv('SRC_DIR'), "data/prices_csv/stocks/*.csv"))

random.seed(42) # Set a seed for reproducibility
# Randomly sample 60 stock price files from the list of available files
stock_files = random.sample(stock_files, 60) 

# Read the stock price files and concatenate them into a single DataFrame

dt_list = [] # Initialize an empty list to hold DataFrames
for s_file in stock_files: # Iterate over each stock price file
    _logs.info(f"Reading file: {s_file}") # Log the file being read
    
    
    dt = pd.read_csv(s_file).assign(                # Add new columns to the DataFrame
        source = os.path.basename(s_file),          # Extract the base name of the file
        ticker = os.path.basename(s_file).replace('.csv', ''), # Extract the ticker symbol from the file name
        Date = lambda x: pd.to_datetime(x['Date'])  # Convert the 'Date' column to datetime format
    ) # Read the CSV file into a DataFrame and assign new columns
    dt_list.append(dt) # Append the DataFrame to the list
stock_prices = pd.concat(dt_list, axis = 0, ignore_index = True)    # Concatenate all DataFrames in the list into a single DataFrame

# pd.concat(...): This is the main pandas function for concatenation, which means joining or linking things together. It's a powerful tool for combining multiple DataFrames.
# dt_list: This is the primary input. It's expected to be a list (or another iterable) of pandas DataFrames. For example, dt_list might look like [df1, df2, df3], where each element is a DataFrame.

# axis=0: This parameter tells concat how to join the DataFrames.
# axis=0 means stack along the rows (the "0-axis"). This is the default behavior.
# axis=1 would mean join along the columns, placing them side-by-side.

# ignore_index=True: This is a crucial step for creating a clean result. By default, concat keeps the original index from each DataFrame. This often results in duplicate index labels.
# When ignore_index=True, the function discards the original indexes and creates a brand new, clean index for the combined DataFrame, starting from 0, 1, 2, ... and so on.

2025-08-22 10:31:26,317, 2962826202.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks\TNC.csv
2025-08-22 10:31:26,351, 2962826202.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks\CBB.csv
2025-08-22 10:31:26,379, 2962826202.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks\ALDX.csv
2025-08-22 10:31:26,389, 2962826202.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks\GLADD.csv
2025-08-22 10:31:26,395, 2962826202.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks\FIXX.csv
2025-08-22 10:31:26,404, 2962826202.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks\ETJ.csv
2025-08-22 10:31:26,418, 2962826202.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks\CMCTP.csv
2025-08-22 10:31:26,423, 2962826202.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks\BWG.csv
2025-08-22 10:31:26,435, 2962826202.py, 13, INFO, Reading file: ../../05_src/data/prices_csv/stocks\VIAC.csv
2025-08-22 10:31:26,4

![alt text](Gemini_Generated_Image_9l0xow9l0xow9l0x.jpeg)

This code finds all stock price CSV files in a specific directory, randomly selects 60 of them, reads each one into a pandas DataFrame while adding new columns for context, and finally combines all of them into a single, large DataFrame.

## 1. Finding and Sampling Files 📂
First, the code locates all the necessary data files and then takes a random sample for processing.

stock_files = glob(...): The glob function searches for files that match a specific pattern. Here, it's looking for any file ending with .csv inside the data/prices_csv/stocks/ directory. The result, stock_files, is a list of all the file paths it finds.

random.seed(42): This line initializes the random number generator with a specific number (a "seed"). It ensures that every time you run this code, the exact same "random" sample will be chosen. This is crucial for making experiments reproducible.

random.sample(stock_files, 60): This command randomly selects 60 unique file paths from the full stock_files list. This is useful for testing your code on a smaller subset of data without having to process everything.

## 2. Reading and Enhancing the Data 🔄
Next, the code loops through the 60 selected file paths to read and process them one by one.

for s_file in stock_files:: This starts a loop that will execute the indented code for each of the 60 file paths.

dt = pd.read_csv(s_file).assign(...): This is the core processing step for each file.

pd.read_csv(s_file): This reads a single CSV file into a pandas DataFrame named dt.

.assign(...): This is a powerful pandas method that allows you to add new columns to the DataFrame in a single step. Here, it's adding three new columns:

source: The name of the original file (e.g., 'AAPL.csv'). This is useful for tracing data back to its origin.

ticker: The stock ticker (e.g., 'AAPL'). This is created by taking the filename and removing the .csv extension.

Date: The original 'Date' column is converted from plain text into a proper datetime format, which is essential for any time-series analysis.

dt_list.append(dt): After a file is read and enhanced, the resulting DataFrame dt is added to the dt_list. After the loop finishes, dt_list will contain 60 individual DataFrames.

## 3. Combining into a Single DataFrame 📚
Finally, the code takes all the individual DataFrames stored in dt_list and merges them into one.

stock_prices = pd.concat(...): As explained previously, this function takes the list of 60 DataFrames and stacks them vertically (axis=0) into a single, master DataFrame called stock_prices. The ignore_index=True argument creates a new, clean index for this combined DataFrame.

In [5]:
len(s_file)

43

In [10]:
os.getenv('s_file')

In [8]:
pd.read_csv(s_file)

,Date,Open,High,Low,Close,Adj Close,Volume
0,1973-02-22,4.250000,4.250000,4.250000,4.250000,0.507389,800
1,1973-02-23,4.250000,4.250000,4.250000,4.250000,0.507389,800
2,1973-02-26,4.250000,4.250000,4.250000,4.250000,0.507389,800
3,1973-02-27,4.250000,4.250000,4.250000,4.250000,0.507389,0
4,1973-02-28,4.250000,4.250000,4.250000,4.250000,0.507389,0
...,...,...,...,...,...,...,...
11878,2020-03-26,54.060001,57.869999,54.060001,57.610001,57.610001,103300
11879,2020-03-27,55.000000,55.369999,52.320000,53.450001,53.450001,72800
11880,2020-03-30,53.810001,58.240002,53.180000,58.020000,58.020000,92200
11881,2020-03-31,57.270000,59.790001,55.880001,57.950001,57.950001,136700


In [6]:
s_file = stock_files[0]
dt = pd.read_csv(s_file).assign(                # Add new columns to the DataFrame
    source = os.path.basename(s_file),          # Extract the base name of the file
    ticker = os.path.basename(s_file).replace('.csv', ''), # Extract the ticker symbol from the file name
    Date = lambda x: pd.to_datetime(x['Date'])  # Convert the 'Date' column to datetime format 
                                                # lambda instead of pd.to_datetime(dt['Date']) to avoid SettingWithCopyWarning 
                                                # when working in a loop
                                                # 
)
dt.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11883 entries, 0 to 11882
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   Date       11883 non-null  datetime64[ns]
 1   Open       11883 non-null  float64       
 2   High       11883 non-null  float64       
 3   Low        11883 non-null  float64       
 4   Close      11883 non-null  float64       
 5   Adj Close  11883 non-null  float64       
 6   Volume     11883 non-null  int64         
 7   source     11883 non-null  object        
 8   ticker     11883 non-null  object        
dtypes: datetime64[ns](1), float64(5), int64(1), object(2)
memory usage: 835.6+ KB


In [11]:
dt_list
# [ ] means list

[            Date       Open       High        Low      Close  Adj Close  \
 0     1973-02-22   4.250000   4.250000   4.250000   4.250000   0.507389   
 1     1973-02-23   4.250000   4.250000   4.250000   4.250000   0.507389   
 2     1973-02-26   4.250000   4.250000   4.250000   4.250000   0.507389   
 3     1973-02-27   4.250000   4.250000   4.250000   4.250000   0.507389   
 4     1973-02-28   4.250000   4.250000   4.250000   4.250000   0.507389   
 ...          ...        ...        ...        ...        ...        ...   
 11878 2020-03-26  54.060001  57.869999  54.060001  57.610001  57.610001   
 11879 2020-03-27  55.000000  55.369999  52.320000  53.450001  53.450001   
 11880 2020-03-30  53.810001  58.240002  53.180000  58.020000  58.020000   
 11881 2020-03-31  57.270000  59.790001  55.880001  57.950001  57.950001   
 11882 2020-04-01  55.139999  55.139999  51.110001  51.919998  51.919998   
 
        Volume   source ticker  
 0         800  TNC.csv    TNC  
 1         800  TNC.

In [15]:
stock_prices.head()


,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker
0,1973-02-22,4.25,4.25,4.25,4.25,0.507389,800.0,TNC.csv,TNC
1,1973-02-23,4.25,4.25,4.25,4.25,0.507389,800.0,TNC.csv,TNC
2,1973-02-26,4.25,4.25,4.25,4.25,0.507389,800.0,TNC.csv,TNC
3,1973-02-27,4.25,4.25,4.25,4.25,0.507389,0.0,TNC.csv,TNC
4,1973-02-28,4.25,4.25,4.25,4.25,0.507389,0.0,TNC.csv,TNC


Verify the structure of the `stock_prices` data:

In [12]:
stock_prices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239659 entries, 0 to 239658
Data columns (total 9 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   Date       239659 non-null  datetime64[ns]
 1   Open       239656 non-null  float64       
 2   High       239656 non-null  float64       
 3   Low        239656 non-null  float64       
 4   Close      239656 non-null  float64       
 5   Adj Close  239656 non-null  float64       
 6   Volume     239656 non-null  float64       
 7   source     239659 non-null  object        
 8   ticker     239659 non-null  object        
dtypes: datetime64[ns](1), float64(6), object(2)
memory usage: 16.5+ MB


We can subset our ticker data set using standard indexing techniques. A good reference for this type of data manipulation is Panda's [Documentation](https://pandas.pydata.org/docs/user_guide/indexing.html#indexing-and-selecting-data) and [Cookbook](https://pandas.pydata.org/docs/user_guide/cookbook.html#cookbook-selection).

From the subset data frame, select one column and convert to list.

In [16]:
select_tickers = stock_prices['ticker'].unique().tolist()
select_tickers

['TNC',
 'CBB',
 'ALDX',
 'GLADD',
 'FIXX',
 'ETJ',
 'CMCTP',
 'BWG',
 'VIAC',
 'REI',
 'BLPH',
 'SMG',
 'MOH',
 'AMH',
 'AMAL',
 'BPYPN',
 'ERH',
 'FAMI',
 'PFG',
 'SPXC',
 'ALL',
 'RTTR',
 'EARN',
 'ZIXI',
 'TSN',
 'WST',
 'REG',
 'MNK',
 'ESGR',
 'NGD',
 'SLRX',
 'GLW',
 'ACN',
 'CSSE',
 'WORK',
 'MOS',
 'IPWR',
 'GLUU',
 'CRMT',
 'EOLS',
 'INSU',
 'BWEN',
 'BPMX',
 'LH',
 'BRQS',
 'KALU',
 'ITCB',
 'SRE',
 'GAZ',
 'AQMS',
 'NPK',
 'QRHC',
 'CGEN',
 'LEVL',
 'BGS',
 'RIV',
 'GURE',
 'TEF',
 'SYNH',
 'KEY']

# Storing Data in CSV



+ We have some data. How do we store it?
+ We can compare two options, CSV and Parqruet, by measuring their performance:

    - Time to save.
    - Space required.

In [ ]:
# Function to get the total size of files in a directory and its subdirectories
def get_dir_size(path='.'): 
    '''Returns the total size of files contained in path.'''
    total = 0
    with os.scandir(path) as it: 
        for entry in it:
            if entry.is_file():
                total += entry.stat().st_size # Add the size of the file to the total
            elif entry.is_dir():
                total += get_dir_size(entry.path) 
                # Recursively needs an exit condition to avoid infinite recursion
                # If the entry is a directory, call get_dir_size on that directory
    return total                # Return the total size of files in bytes
# exit(0) # Exit the script with a success status code


In [ ]:
import time
import shutil #This deletes files and directories


In [24]:
temp = os.getenv("TEMP_DATA")
csv_dir = os.path.join(temp, "csv")
shutil.rmtree(csv_dir, ignore_errors=True)
stock_csv = os.path.join(csv_dir, "stock_px.csv")
os.makedirs(csv_dir, exist_ok=True)

In [20]:
os.path.join(csv_dir, "stock_px.csv")

'../../05_src/data/temp/csv\\stock_px.csv'

In [21]:

start = time.time()
stock_prices.to_csv(stock_csv, index = False)
end = time.time()

_logs.info(f'Writing data ({stock_prices.shape}) to csv took {end - start} seconds.')
_logs.info(f'CSV file size { os.path.getsize(stock_csv)*1e-6 } MB')

2025-08-22 11:33:54,919, 473192941.py, 5, INFO, Writing data ((239659, 9)) to csv took 2.232999086380005 seconds.
2025-08-22 11:33:54,922, 473192941.py, 6, INFO, CSV file size 26.618403999999998 MB


## Save Data to Parquet

### Dask 

We can work with with large data sets and parquet files. In fact, recent versions of pandas support pyarrow data types and future versions will require a pyarrow backend. The pyarrow library is an interface between Python and the Appache Arrow project. The [parquet data format](https://parquet.apache.org/) and [Arrow](https://arrow.apache.org/docs/python/parquet.html) are projects of the Apache Foundation.

However, Dask is much more than an interface to Arrow: Dask provides parallel and distributed computing on pandas-like dataframes. It is also relatively easy to use, bridging a gap between pandas and Spark. 

In [25]:
import dask.dataframe as dd

parquet_dir = os.path.join(temp, "parquet") # Define the directory path for Parquet files
shutil.rmtree(parquet_dir, ignore_errors=True) # Remove the directory and its contents if it exists
os.makedirs(parquet_dir, exist_ok=True) # Create the directory if it doesn't exist

In [26]:
px_dd = dd.from_pandas(stock_prices, npartitions = len(select_tickers))

start = time.time()
px_dd.to_parquet(parquet_dir, engine = "pyarrow")
end = time.time()

_logs.info(f'Writing dd ({stock_prices.shape}) to parquet took {end - start} seconds.')
_logs.info(f'Parquet file size { get_dir_size(parquet_dir)*1e-6 } MB')

2025-08-22 12:45:41,662, 817812245.py, 7, INFO, Writing dd ((239659, 9)) to parquet took 1.432992696762085 seconds.
2025-08-22 12:45:41,668, 817812245.py, 8, INFO, Parquet file size 9.713265 MB


### Parquet files and Dask Dataframes

+ Parquet files are immutable: once written, they cannot be modified.
+ Dask DataFrames are a useful implementation to manipulate data stored in parquets.
+ Parquet and Dask are not the same: parquet is a file format that can be accessed by many applications and programming languages (Python, R, PowerBI, etc.), while Dask is a package in Python to work with large datasets using distributed computation.
+ **Dask is not for everything** (see [Dask DataFrames Best Practices](https://docs.dask.org/en/stable/dataframe-best-practices.html)). 

    - Consider cases suchas small to large joins, where the small dataframe fits in memory, but the large one does not. 
    - If possible, use pandas: reduce, then use pandas.
    - Pandas performance tips apply to Dask.
    - Use the index: it is beneficial to have a well-defined index in Dask DataFrames, as it may speed up searching (filtering) the data. A one-dimensional index is allowed.
    - Avoid (or minimize) full-data shuffling: indexing is an expensive operations. 
    - Some joins are more expensive than others. 

        * Not expensive:

            - Join a Dask DataFrame with a pandas DataFrame.
            - Join a Dask DataFrame with another Dask DataFrame of a single partition.
            - Join Dask DataFrames along their indexes.

        * Expensive:

            - Join Dask DataFrames along columns that are not their index.


# How do we store prices?

+ We can store our data as a single blob. This can be difficult to maintain, especially because parquet files are immutable.
+ Strategy: organize data files by ticker and date. Update only latest month.



In [27]:
# CLean up before start
PRICE_DATA = os.getenv("PRICE_DATA")
import shutil
if os.path.exists(PRICE_DATA):
    shutil.rmtree(PRICE_DATA)

In [28]:
stock_prices.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'source',
       'ticker'],
      dtype='object')

In [29]:
stock_prices['ticker'].unique()

array(['TNC', 'CBB', 'ALDX', 'GLADD', 'FIXX', 'ETJ', 'CMCTP', 'BWG',
       'VIAC', 'REI', 'BLPH', 'SMG', 'MOH', 'AMH', 'AMAL', 'BPYPN', 'ERH',
       'FAMI', 'PFG', 'SPXC', 'ALL', 'RTTR', 'EARN', 'ZIXI', 'TSN', 'WST',
       'REG', 'MNK', 'ESGR', 'NGD', 'SLRX', 'GLW', 'ACN', 'CSSE', 'WORK',
       'MOS', 'IPWR', 'GLUU', 'CRMT', 'EOLS', 'INSU', 'BWEN', 'BPMX',
       'LH', 'BRQS', 'KALU', 'ITCB', 'SRE', 'GAZ', 'AQMS', 'NPK', 'QRHC',
       'CGEN', 'LEVL', 'BGS', 'RIV', 'GURE', 'TEF', 'SYNH', 'KEY'],
      dtype=object)

In [32]:
stock_prices[stock_prices['ticker'] == 'TNC']
# simpliest way to filter a DataFrame by a specific column value
# This line filters the stock_prices DataFrame to include only rows where the 'ticker' column

,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker
0,1973-02-22,4.250000,4.250000,4.250000,4.250000,0.507389,800.0,TNC.csv,TNC
1,1973-02-23,4.250000,4.250000,4.250000,4.250000,0.507389,800.0,TNC.csv,TNC
2,1973-02-26,4.250000,4.250000,4.250000,4.250000,0.507389,800.0,TNC.csv,TNC
3,1973-02-27,4.250000,4.250000,4.250000,4.250000,0.507389,0.0,TNC.csv,TNC
4,1973-02-28,4.250000,4.250000,4.250000,4.250000,0.507389,0.0,TNC.csv,TNC
...,...,...,...,...,...,...,...,...,...
11878,2020-03-26,54.060001,57.869999,54.060001,57.610001,57.610001,103300.0,TNC.csv,TNC
11879,2020-03-27,55.000000,55.369999,52.320000,53.450001,53.450001,72800.0,TNC.csv,TNC
11880,2020-03-30,53.810001,58.240002,53.180000,58.020000,58.020000,92200.0,TNC.csv,TNC
11881,2020-03-31,57.270000,59.790001,55.880001,57.950001,57.950001,136700.0,TNC.csv,TNC


In [33]:
for ticker in stock_prices['ticker'].unique():
    ticker_dt = stock_prices[stock_prices['ticker'] == ticker]
    ticker_dt = ticker_dt.assign(Year = ticker_dt.Date.dt.year)
    for yr in ticker_dt['Year'].unique():
        yr_dd = dd.from_pandas(ticker_dt[ticker_dt['Year'] == yr],2)
        yr_path = os.path.join(PRICE_DATA, ticker, f"{ticker}_{yr}")
        os.makedirs(os.path.dirname(yr_path), exist_ok=True)
        yr_dd.to_parquet(yr_path, engine = "pyarrow")
    

Why would we want to store data this way?

+ Easier to maintain. We do not update old data, only recent data.
+ We can also access all files as follows.

# Load, Transform and Save 

## Load

+ Parquet files can be read individually or as a collection.
+ `dd.read_parquet()` can take a list (collection) of files as input.
+ Use `glob` to get the collection of files.

In [ ]:
from glob import glob

parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
dd_px = dd.read_parquet(parquet_files).set_index("ticker")



,Date,Open,High,Low,Close,Adj Close,Volume,source,Year
ticker,,,,,,,,,
ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001
ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001
ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001
ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001
ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001


In [38]:
glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)

['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2003\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2004\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2004\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2005\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2005\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2006\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2006\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2007\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2007\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2008\\part.0.parquet',
 '../../05_src/data/prices\\ACN\\ACN_2008\\part.1.parquet',
 '../../05_src/data/prices\\ACN\\ACN_200

In [39]:
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True)
len(parquet_files)

2045

In [40]:
dd_px

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year
npartitions=60,,,,,,,,,
ACN,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32
ALDX,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...


## Transform

+ This transformation step will create a *Features* data set. In our case, features will be stock returns (we obtained prices).
+ Dask dataframes work like pandas dataframes: in particular, we can perform groupby and apply operations.
+ Notice the use of [an anonymous (lambda) function](https://realpython.com/python-lambda/) in the apply statement.

In [ ]:
dd_shift = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1))
)
#groupby('ticker', group_keys=False): This part of the code groups the Dask DataFrame (dd_px) by the 'ticker' column. Grouping is a common operation in data analysis that allows you to perform operations on subsets of data that share a common value in a specified column. In this case, all rows with the same ticker symbol are grouped together.
#group_keys=False: This parameter is used to control whether the group keys (in this case, the unique values in the 'ticker' column) are included in the output. Setting group_keys

# Cloose_lag_1 = x['Close'].shift(1): This part of the code defines a lambda function that is applied to each group created by the groupby operation. The lambda function takes a DataFrame (x) as input and assigns a new column called 'Close_lag_1' to it. The value of this new column is calculated by shifting the 'Close' column down by one row using the shift(1) method. This means that for each row in the DataFrame, 'Close_lag_1' will contain the closing price from the previous row within the same ticker group.
# When group_keys is set to False, the resulting DataFrame will not include the 'ticker' column as part of its index or as a separate column. Instead, the output will be a
# Usually Close_lag_1 will require sorting by date within each ticker group to ensure that the lagged values correspond to the correct previous dates.
# But the way the data is stored in parquet files, it is already be sorted by date within each ticker group.  


C:\Users\Saad Khan\AppData\Local\Temp\ipykernel_11328\102405636.py:1: UserWarning: `meta` is not specified, inferred from partial data.
Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result

  dd_shift = dd_px.groupby('ticker', group_keys=False).apply(


In [42]:
dd_shift

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1
npartitions=60,,,,,,,,,,
ACN,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32,float64
ALDX,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...


In [43]:
dd_rets = dd_shift.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1
)

## Lazy Exection

What does `dd_rets` contain?

In [44]:
dd_rets

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Returns
npartitions=60,,,,,,,,,,,
ACN,datetime64[ns],float64,float64,float64,float64,float64,float64,string,int32,float64,float64
ALDX,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...


+ Dask is a lazy execution framework: commands will not execute until they are required. 
+ To trigger an execution in dask use `.compute()`.

In [45]:
dd_rets.compute()

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Returns
ticker,,,,,,,,,,,
ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001,NaN,NaN
ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001,15.17,-0.010547
ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001,15.01,-0.000666
ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001,15.00,-0.009333
ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001,14.86,0.006057
...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,2003-06-26,4.04,4.19,3.86,4.00,4.000000,515300.0,ZIXI.csv,2003,4.04,-0.009901
ZIXI,2003-06-27,4.00,4.05,3.79,3.85,3.850000,162400.0,ZIXI.csv,2003,4.00,-0.037500
ZIXI,2003-06-30,3.84,4.00,3.72,3.77,3.770000,119900.0,ZIXI.csv,2003,3.85,-0.020779


## Save

+ Apply transformations to calculate daily returns
+ Store the enriched data, the silver dataset, in a new directory.
+ Should we keep the same namespace? All columns?

In [46]:
# CLean up before save
FEATURES_DATA = os.getenv("FEATURES_DATA")
if os.path.exists(FEATURES_DATA):
    shutil.rmtree(FEATURES_DATA)
dd_rets.to_parquet(FEATURES_DATA, overwrite = True)

In [47]:
os.getenv("FEATURES_DATA")

'../../05_src/data/features/stock_features'

# Optional: from Jupyter to Command Line

+ We have drafted our code in a Jupyter Notebook. 
+ Finalized code should be written in Python modules.

## Object Oriented vs Functional Programming

+ We can use classes to keep parameters and functions together.
+ We *could* use Object Oriented Programming, but parallelization of data manipulation and modelling tasks benefit from *Functional Programming*.
+ An Idea: 

    - [Data Oriented Programming](https://blog.klipse.tech/dop/2022/06/22/principles-of-dop.html).
    - Use the class to bundle together parameters and functions.
    - Use stateless operations and treat all data objects as immutable (we do not modify them, we overwrite them).
    - Take advantage of [`@staticmethod`](https://realpython.com/instance-class-and-static-methods-demystified/).

The code is in `./05_src/stock_prices/data_manager.py`.

Our original design was:

![](./images/02_target_pipeline_manager.png)



In [ ]:
from stock_prices.data_manager import DataManager
dm = DataManager()

Download all prices.

In [ ]:
dm.process_sample_files()

Finally, add features to the data set and save to a *feature store*.

In [ ]:
dm.featurize()